# 第09章 图像梯度（跟随代码）

> 《OpenCV轻松入门：面向Python》（李立宗）第 9 章跟随代码。**全量收录原书代码**（例 9.1~例 9.14 + 章内独立演示），缺失例题已按书补全，代码以网页笔记 ch09.md 为准（含笔误与参数修正，如 Sobel 启用 Scharr 必须写 `ksize=-1` 关键字）。
> `cv2.imshow` 需在**本地 Python 环境**运行（浏览器/Jupyter 内不弹窗）；运行前请确认**工作目录为素材根目录**（本文件夹，picture*.jpg/png 等素材都在这里），否则 `imread` 返回 None。
> 原 notebook 用到的 `cv_show` 是未给出定义的辅助函数，已在本 notebook 开头补上定义（本地运行显示窗口用）。

# 第九章 图像梯度

> 图像梯度衡量灰度变化的速度：边缘处灰度突变、梯度值大，平滑处接近 0；图像上不求真正的导数，用**像素差分**近似。Sobel＝高斯平滑＋一阶差分，Scharr 是它的高精度 3×3 版，Laplacian 是二阶导数、旋转不变。实操三件事：`ddepth` 用 `cv2.CV_64F` 防负数截断、`cv2.convertScaleAbs` 转回 8 位、`cv2.addWeighted` 叠加两个方向的边缘。

In [ ]:
# cv_show 辅助函数定义（本地运行显示窗口用）：原 notebook 直接使用了它但未给出定义
def cv_show(name, img):
    cv2.imshow(name, img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

import cv2  # 供 cv_show 与后续示例使用

## Sobel 算子

`dst = cv2.Sobel(src, ddepth, dx, dy[, ksize[, scale[, delta[, borderType]]]])`
- `src`：输入图像，单通道或多通道
- `ddepth`：输出图像的深度，**建议 `cv2.CV_64F`**（`CV_16S`/`CV_32F` 也可保留负值；设 -1 时负梯度被截断为 0，丢一侧边界）
- `dx`：x 方向上的**求导阶数**（通常取 0 或 1，最大 2；与 dy 不能同时为 0）
- `dy`：y 方向上的求导阶数（同上）
- `ksize`：Sobel 核大小，取 1、3、5、7；**设为 -1 时使用 Scharr 算子——必须写成关键字 `ksize=-1`**（位置传参 `Sobel(...,1,0,-1)` 的 -1 会落到第 5 个参数 dst 占位上，实际等于普通 Sobel）
- `scale`：计算导数值时的缩放因子，默认 1
- `delta`：加在目标图像 dst 上的值，默认 0（不是阈值）
- `borderType`：边界样式，默认 `cv2.BORDER_DEFAULT`（= `BORDER_REFLECT_101`）

$$
dst=saturate(src\times kernel\times scale+delta)
$$

为什么不能直接用 cv2.CV_8U (或 -1)？
原因：Sobel算子是在做减法。
从黑(0)到白(255)的边缘，导数是正数。
从白(255)到黑(0)的边缘，导数是负数。
如果你设定输出深度为 uint8，所有的负数会被OpenCV自动截断为0（就像你之前遇到的黑帽运算问题一样）。这意味着你会丢失所有由亮变暗的边缘！
正确做法：使用 cv2.CV_64F (64位浮点数) 或 cv2.CV_16S (16位有符号整数) 来保留负梯度，最后再取绝对值转回 uint8。

**代码导览**：例 9.1（convertScaleAbs 取绝对值）→ 独立演示（手写 Sobel 核 + 梯度幅值）→ 例 9.2（ddepth=-1 的陷阱）→ 例 9.3~9.6（x 方向 / y 方向 / dx=1,dy=1 / addWeighted 叠加，合并在一个 cell）→ 例 9.7（两种合成方式对比）。

### 9.2.1 convertScaleAbs：取绝对值并转回 8 位图

`dst = cv2.convertScaleAbs(src[, alpha[, beta]])`，即 `dst = saturate(src × alpha + beta)`：负数变正数，超过 255 饱和为 255（**饱和**，区别于 `astype(np.uint8)` 的内存回绕），结果转回 8 位无符号图。

In [ ]:
# ===== 9.2.1 例 9.1：convertScaleAbs 取绝对值 =====
import cv2
import numpy as np

img = np.random.randint(-256, 256, size=[4, 5], dtype=np.int16)  # -256~255 的随机 int16 数组
rst = cv2.convertScaleAbs(img)              # 转为 8 位无符号：负数变正数，超界饱和
print("img=\n", img)
print("rst=\n", rst)

#### 独立演示 · 手写 Sobel 核 + 梯度幅值合成

用 `filter2D` 分别卷 Gx、Gy 两个 Sobel 核，比较 √(Gx²+Gy²)（梯度幅值）与 |Gx|+|Gy|（分块线性近似）两种合成，取较大者并裁剪到 [0,255]——比 0.5/0.5 加权叠加更接近"真正的梯度幅值"。

In [11]:
#使用梯度增强边缘
img=cv2.imread('picture2.png',0)
#手动锐化
Gx=np.array([[-1,0,1],[-2,0,2],[-1,0,1]])
Gy=np.array([[-1,-2,-1],[0,0,0],[1,2,1]])
img_x=cv2.filter2D(img,cv2.CV_64F,Gx)
img_y=cv2.filter2D(img,cv2.CV_64F,Gy)
img_xy1=np.sqrt(img_x**2+img_y**2)
img_xy2=np.abs(img_x)+np.abs(img_y)
img_xy=np.maximum(img_xy1,img_xy2)
img_xy=np.clip(img_xy,0,255).astype(np.uint8)
cv_show("img",img)
cv_show("img_xy",img_xy)

### 例 9.2 · ddepth=-1 的陷阱

`ddepth=-1` 时结果仍是 8 位图：左边界在运算时得到负值、显示时被截为 0，只剩字母 j 的右边界。要完整边缘必须 `CV_64F` + 取绝对值（见例 9.3）。

随后一个代码 cell 依次演示：**例 9.3**（dx=1,dy=0，CV_64F + convertScaleAbs，水平方向完整边缘）、**例 9.4**（dx=0,dy=1，垂直方向完整边缘）、**例 9.5**（dx=1,dy=1，混合偏导数，只剩零星角点白点）、**例 9.6**（addWeighted 0.5/0.5 叠加两个方向，得完整边缘）。

In [ ]:
# ===== 9.2.3 例 9.2：ddepth=-1 的陷阱 =====
import cv2

o = cv2.imread('picture7.png', cv2.IMREAD_GRAYSCALE)   # 原书 Sobel4.bmp
Sobelx = cv2.Sobel(o, -1, 1, 0)                        # ddepth=-1：结果仍是 8 位图
cv2.imshow("original", o)
cv2.imshow("x", Sobelx)
cv2.waitKey()
cv2.destroyAllWindows()

In [237]:
#计算x方向边缘梯度：dx=1,dy=0

img=cv2.imread("picture7.png",cv2.IMREAD_GRAYSCALE)
dst1=cv2.Sobel(img,cv2.CV_64F,dx=1,dy=0)
dst1=cv2.convertScaleAbs(dst1)
cv_show("Sobel",dst1)

#计算y方向边缘梯度：dx=0,dy=1

dst2=cv2.Sobel(img,cv2.CV_64F,dx=0,dy=1)
dst2=cv2.convertScaleAbs(dst2)
cv_show("Sobel",dst2)

#计算x方向边缘梯度和y方向边缘梯度的和：dx=1,dy=1

dst3=cv2.Sobel(img,cv2.CV_64F,dx=1,dy=1)
dst3=cv2.convertScaleAbs(dst3)
cv_show("Sobel",dst3)

dst4=cv2.addWeighted(dst1,0.5,dst2,0.5,0)
cv_show("Sobel",dst4)

### 例 9.7 · 两种合成方式对比

方式 1：分别计算 dx=1,dy=0 与 dx=0,dy=1 后用 addWeighted 相加——完整边缘；方式 2：直接 dx=1,dy=1——混合偏导数，只剩微弱响应、画面明显偏暗。要拿"完整边缘"，就走"分方向计算 + addWeighted 叠加"的路。

In [ ]:
# ===== 9.2.3 例 9.7：两种合成方式对比 =====
import cv2

o = cv2.imread('picture1.jpg', cv2.IMREAD_GRAYSCALE)   # 原书 lena.bmp

# 方式 1：分方向计算后再叠加
Sobelx = cv2.Sobel(o, cv2.CV_64F, 1, 0)
Sobely = cv2.Sobel(o, cv2.CV_64F, 0, 1)
Sobelx = cv2.convertScaleAbs(Sobelx)
Sobely = cv2.convertScaleAbs(Sobely)
Sobelxy = cv2.addWeighted(Sobelx, 0.5, Sobely, 0.5, 0)

# 方式 2：dx=1, dy=1 一步到位
Sobelxy11 = cv2.Sobel(o, cv2.CV_64F, 1, 1)
Sobelxy11 = cv2.convertScaleAbs(Sobelxy11)

cv2.imshow("original", o)
cv2.imshow("xy", Sobelxy)          # 方式 1：完整边缘
cv2.imshow("xy11", Sobelxy11)      # 方式 2：明显偏暗
cv2.waitKey()
cv2.destroyAllWindows()

## Scharr 与 Sobel 的区别

- 卷积核权重：Scharr 核平滑权重为 3、10、3，Sobel 核为 1、2、1。
- 精度：同样 3×3 核、同样速度，Scharr 精度更高（把 3×3 差分核的旋转对称性误差做到最小）。
- 噪声敏感性：Sobel 最低（[1,2,1] 平滑成分权重最大）、Scharr 居中、Laplacian 最高（无平滑成分）。
- 应用：Scharr 适合 3×3 核下的高精度梯度；Sobel 适合通用边缘检测，不需要极高精度时更抗噪。
- 尺寸限制：Scharr 核尺寸只能为 3×3；Sobel 核尺寸为 1、3、5、7（ksize=-1 时退让给 Scharr）。

## Scharr 算子

`dst = cv2.Scharr(src, ddepth, dx, dy[, scale[, delta[, borderType]]])`
- `src`：输入图像，单通道或多通道
- `ddepth`：输出图像深度，含义与 `cv2.Sobel()` 中相同——**建议 `cv2.CV_64F` 并配合 `convertScaleAbs` 取绝对值**（设 -1 会截断负梯度）
- `dx`/`dy`：x / y 方向上的求导阶数，约束 `dx >= 0 && dy >= 0 && dx+dy == 1`——**不允许 dx=1, dy=1**（与 Sobel 的实际差别）
- `scale`：缩放因子，默认 1
- `delta`：加到目标图像上的亮度值，默认 0
- `borderType`：边界填充方式，默认 `cv2.BORDER_DEFAULT`

核通常为 Gx=[-3,0,3; -10,0,10; -3,0,3]、Gy=[-3,-10,-3; 0,0,0; 3,10,3]。`cv2.Scharr(src,ddepth,dx,dy)` 与 `cv2.Sobel(src,ddepth,dx,dy,ksize=-1)` 等价——**ksize 必须关键字传参**（位置写法的 -1 会落到第 5 个参数 dst 占位上，等于普通 Sobel）。两个方向的边缘叠加：先分别算 dx=1,dy=0 与 dx=0,dy=1，各自 convertScaleAbs 后 `cv2.addWeighted(x,0.5,y,0.5,0)`。

下方第一个代码 cell 合并演示**例 9.8**（水平方向边缘）、**例 9.9**（垂直方向边缘）、**例 9.10**（两方向叠加）；其后是 CV_16S 深度的独立演示，再往后为例 9.11、例 9.12。

In [ ]:
# ===== 9.3 例 9.8~9.10：Scharr 三个方向的边缘 =====
#计算x方向边缘梯度：dx=1,dy=0

img=cv2.imread("picture7.png",cv2.IMREAD_GRAYSCALE)   # 原书 Scharr.bmp
dst1=cv2.Scharr(img,cv2.CV_64F,dx=1,dy=0)
dst1=cv2.convertScaleAbs(dst1)
cv_show("Scharr x",dst1)

#计算y方向边缘梯度：dx=0,dy=1

dst2=cv2.Scharr(img,cv2.CV_64F,dx=0,dy=1)
dst2=cv2.convertScaleAbs(dst2)
cv_show("Scharr y",dst2)

#水平方向和垂直方向边缘叠加

dst4=cv2.addWeighted(dst1,0.5,dst2,0.5,0)
cv_show("Scharr xy",dst4)

### 例 9.11 · dx=1, dy=1 直接报错

Scharr 要求 `dx >= 0 && dy >= 0 && dx+dy == 1`，把两个参数都设为 1 后程序报错 `Assertion failed ... getScharrKernels`。Sobel 允许 dx=1, dy=1，Scharr 不允许——这是两个函数参数约束上的实际差别。

In [ ]:
# ===== 9.3 例 9.11：dx=1, dy=1 直接报错 =====
import cv2

o = cv2.imread('picture7.png', cv2.IMREAD_GRAYSCALE)   # 原书此处路径为 image\Scharr.bmp
Scharrxy11 = cv2.Scharr(o, cv2.CV_64F, 1, 1)           # 违反 dx+dy==1 约束
cv2.imshow("original", o)
cv2.imshow("xy11", Scharrxy11)
cv2.waitKey()
cv2.destroyAllWindows()

### 例 9.12 · 用 Sobel 完成 Scharr 运算（ksize=-1）

`cv2.Sobel()` 的 ksize 为 -1 时使用 Scharr 算子进行运算。**必须写成关键字 `ksize=-1`**：原书的位置写法 `(…,1,0,-1)` 在 Python 绑定中会把 -1 传给第 5 个参数 dst（输出占位），实际等于普通 Sobel 而非 Scharr。

In [ ]:
# ===== 9.3 例 9.12：Sobel 的 ksize=-1 形式 =====
import cv2

o = cv2.imread('picture7.png', cv2.IMREAD_GRAYSCALE)   # 原书 Sobel4.bmp
Scharrx = cv2.Sobel(o, cv2.CV_64F, 1, 0, ksize=-1)     # ksize=-1 即 Scharr；原书位置写法 (…,1,0,-1) 的 -1 会落到 dst 上
Scharry = cv2.Sobel(o, cv2.CV_64F, 0, 1, ksize=-1)
Scharrx = cv2.convertScaleAbs(Scharrx)
Scharry = cv2.convertScaleAbs(Scharry)
cv2.imshow("original", o)
cv2.imshow("x", Scharrx)
cv2.imshow("y", Scharry)
cv2.waitKey()
cv2.destroyAllWindows()

In [7]:
img=cv2.imread("picture1.jpg",cv2.IMREAD_GRAYSCALE)

#方法1
dst1=cv2.Scharr(img,cv2.CV_16S,1,0,scale=1)
dst2=cv2.Scharr(img,cv2.CV_16S,0,1,scale=1)

dst1=cv2.convertScaleAbs(dst1)
dst2=cv2.convertScaleAbs(dst2)

dst3=cv2.addWeighted(dst1,0.5,dst2,0.5,0)

cv_show("img",img)
cv_show("dst1",dst1)
cv_show("dst2",dst2)
cv_show("dst3",dst3)

### 例 9.13 · Sobel 与 Scharr 同图对比

对同一幅灰度图分别做 Sobel（ksize=3）与 Scharr 的两方向叠加：Scharr 的边缘响应整体更亮、发丝等弱纹理区域的边缘更多——同样 3×3 大小、同样计算量，Scharr 灵敏度更高。

In [ ]:
# ===== 9.4 例 9.13：Sobel 与 Scharr 同图对比 =====
import cv2

o = cv2.imread('picture1.jpg', cv2.IMREAD_GRAYSCALE)   # 原书 lena.bmp

# Sobel：ksize=3
Sobelx = cv2.Sobel(o, cv2.CV_64F, 1, 0, ksize=3)
Sobely = cv2.Sobel(o, cv2.CV_64F, 0, 1, ksize=3)
Sobelx = cv2.convertScaleAbs(Sobelx)
Sobely = cv2.convertScaleAbs(Sobely)
Sobelxy = cv2.addWeighted(Sobelx, 0.5, Sobely, 0.5, 0)

# Scharr
Scharrx = cv2.Scharr(o, cv2.CV_64F, 1, 0)
Scharry = cv2.Scharr(o, cv2.CV_64F, 0, 1)
Scharrx = cv2.convertScaleAbs(Scharrx)
Scharry = cv2.convertScaleAbs(Scharry)
Scharrxy = cv2.addWeighted(Scharrx, 0.5, Scharry, 0.5, 0)

cv2.imshow("original", o)
cv2.imshow("Sobelxy", Sobelxy)
cv2.imshow("Scharrxy", Scharrxy)
cv2.waitKey()
cv2.destroyAllWindows()

## Laplacian 算子

Laplacian（拉普拉斯）算子是**二阶导数算子**，具有旋转不变性，可以满足不同方向的图像边缘锐化（边缘检测）要求；算子系数之和需要为零。ksize=1 时用 3×3 核 [0,1,0; 1,-4,1; 0,1,0]，响应 = 四邻域之和 − 4×中心：平滑区响应为 0，边缘处可正可负——**必须取绝对值**才能正确显示。把中心系数改为 -5 的扩展核可用于对比度增强（锐化）。

`dst = cv2.Laplacian(src, ddepth[, ksize[, scale[, delta[, borderType]]]])`
- `ddepth`：建议 `cv2.CV_64F`，配合 `convertScaleAbs` 取绝对值
- `ksize`：计算二阶导数的核尺寸，**必须是正奇数**；默认 1 即上式 3×3 核

### 例 9.14 · Laplacian 边缘

对黑底白色字母图做 Laplacian（默认 ksize=1 的 3×3 核）+ 取绝对值，得到边缘信息。其后一个 cell 是 picture2 上的同款独立演示。

In [ ]:
# ===== 9.5 例 9.14：Laplacian 边缘 =====
import cv2

o = cv2.imread('picture7.png', cv2.IMREAD_GRAYSCALE)   # 原书 Laplacian.bmp
Laplacian = cv2.Laplacian(o, cv2.CV_64F)               # 默认 ksize=1：3×3 核
Laplacian = cv2.convertScaleAbs(Laplacian)             # 二阶导有正有负，取绝对值
cv2.imshow("original", o)
cv2.imshow("Laplacian", Laplacian)
cv2.waitKey()
cv2.destroyAllWindows()

In [9]:
img=cv2.imread("picture2.png",cv2.IMREAD_GRAYSCALE)

laplacian=cv2.Laplacian(img,cv2.CV_64F)
laplacian=cv2.convertScaleAbs(laplacian)

cv_show("Laplacian",laplacian)